In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "ModelSpace/GemmaX2-28-9B-v0.1"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# You can now use the model for inference

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /ModelSpace/GemmaX2-28-9B-v0.1/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ed64ec14980>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 06138d5d-3cc5-4a5c-bd3b-b326fdd14bc9)')' thrown while requesting HEAD https://huggingface.co/ModelSpace/GemmaX2-28-9B-v0.1/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
import re
from typing import Dict, List, Tuple, Optional

def parse_translation_block(text_block: str) -> Dict:
    
    # 1. Extract content and field
    match_field = re.search(r'^(?:"|)(.*?)(?:"|),"(.*?)"$', text_block.strip(), re.DOTALL)
    
    if not match_field:
        return {"field": None, "extracted_terms": [], "few_shot_examples": [], "target_sentence_english": None}
    
    content = match_field.group(1).strip()
    field = match_field.group(2).strip()
    print(f"Extracted field: {field}")
    print(f"Content to parse:\n{content}\n")
    
    # 2. Extracting Terms
    terms_blocks = re.findall(r'Terms: (.*?)\n', content, re.DOTALL)
    extracted_terms: List[Tuple[str, str]] = []
    
    for terms_str in terms_blocks:
        pairs = terms_str.strip().split(' - ')
        for pair in pairs:
            if '=' in pair:
                eng_term, arab_term = pair.split('=', 1)
                if (eng_term.strip(), arab_term.strip()) not in extracted_terms:
                    extracted_terms.append((eng_term.strip(), arab_term.strip()))

    # --- 3. Extract all sentences (complete and incomplete) ---
    
    # Updated pattern to handle:
    # - Complete pairs: English: ... Arabic: <arabic text>
    # - Incomplete pairs: English: ... Arabic: (empty or just whitespace before end)
    all_examples_pattern = re.compile(
        r'English:\s*(.*?)\nArabic:\s*(.*?)(?=\nEnglish:|\nTerms:|$)', re.DOTALL
    )
    
    all_sentences_matches = all_examples_pattern.findall(content)
    print(f"All sentence matches found: {all_sentences_matches}")
    
    few_shot_examples: List[Tuple[str, str]] = []
    target_sentence: Optional[str] = None
    
    # Iterate through all matches
    for eng, arab in all_sentences_matches:
        eng_clean = eng.strip()
        arab_clean = arab.strip()
        
        # If Arabic is empty or just whitespace, this is the target sentence
        if not arab_clean:
            target_sentence = eng_clean
        # Otherwise, it's a complete few-shot example
        else:
            few_shot_examples.append((eng_clean, arab_clean))
    
    return {
        "field": field,
        "extracted_terms": extracted_terms,
        "few_shot_examples": few_shot_examples,
        "target_sentence_english": target_sentence
    }

# --- Example Usage with a new target sentence structure ---

text_input_new = '''
"Terms: forex=العملات الأجنبية - intervention=التدخل - carry trade=تجارة المراجحة
English: Trading volume in the forex market is substantial.
Arabic: حجم التداول في سوق العملات الأجنبية كبير.

English: The central bank conducted intervention to stabilize the exchange rate.
Arabic: قام البنك المركزي بالتدخل لتثبيت سعر الصرف.

English: Floating exchange-rate regime must be monitored by currency intervention mechanism during carry trade.
Arabic:","finance"
'''

# Parse the text block
parsed_data = parse_translation_block(text_input_new)

# Display Results
print("--- Parsing Results (Finance Example) ---")
print(f"Field: {parsed_data.get('field')}")
print("\n--- Extracted Terms (Unique) ---")
for eng, arab in parsed_data.get('extracted_terms', []):
    print(f"English: '{eng}' -> Arabic: '{arab}'")

print("\n--- Few-Shot Examples (Complete) ---")
for eng, arab in parsed_data.get('few_shot_examples', []):
    print(f"English: '{eng}'")
    print(f"Arabic: '{arab}'")

print("\n--- Target Sentence for Translation ---")
print(f"English: '{parsed_data.get('target_sentence_english')}'")

Extracted field: finance
Content to parse:
Terms: forex=العملات الأجنبية - intervention=التدخل - carry trade=تجارة المراجحة
English: Trading volume in the forex market is substantial.
Arabic: حجم التداول في سوق العملات الأجنبية كبير.

English: The central bank conducted intervention to stabilize the exchange rate.
Arabic: قام البنك المركزي بالتدخل لتثبيت سعر الصرف.

English: Floating exchange-rate regime must be monitored by currency intervention mechanism during carry trade.
Arabic:

All sentence matches found: [('Trading volume in the forex market is substantial.', 'حجم التداول في سوق العملات الأجنبية كبير.\n'), ('The central bank conducted intervention to stabilize the exchange rate.', 'قام البنك المركزي بالتدخل لتثبيت سعر الصرف.\n'), ('Floating exchange-rate regime must be monitored by currency intervention mechanism during carry trade.', '')]
--- Parsing Results (Finance Example) ---
Field: finance

--- Extracted Terms (Unique) ---
English: 'forex' -> Arabic: 'العملات الأجنبية'
En

In [4]:
import pandas as pd
import os
from glob import glob

def load_and_parse_csv(csv_path: str) -> List[Dict]:
    """
    Load a CSV file and parse all translation blocks.
    Returns a list of parsed dictionaries.
    """
    # Read CSV file
    df = pd.read_csv(csv_path)
    
    parsed_results = []
    
    for idx, row in df.iterrows():
        query = row['query']
        field = row['field']
        
        # Create the text block in the expected format
        text_block = f'"{query}","{field}"'
        
        # Parse using our existing function
        parsed = parse_translation_block(text_block)
        parsed['row_index'] = idx
        parsed['source_file'] = os.path.basename(csv_path)
        
        parsed_results.append(parsed)
    
    return parsed_results


def load_all_csv_files(folder_path: str) -> Dict[str, List[Dict]]:
    """
    Load and parse all *_examples.csv files from a folder.
    Returns a dictionary with filename as key and list of parsed data as value.
    """
    all_data = {}
    
    # Find all CSV files ending with _examples.csv
    csv_files = glob(os.path.join(folder_path, "*_examples.csv"))
    
    print(f"Found {len(csv_files)} CSV files:")
    for f in csv_files:
        print(f"  - {os.path.basename(f)}")
    print()
    
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        print(f"Processing: {filename}")
        
        parsed_data = load_and_parse_csv(csv_file)
        all_data[filename] = parsed_data
        
        print(f"  Parsed {len(parsed_data)} entries\n")
    
    return all_data


# --- Load all CSV files from the Augmented Queries Samples folder ---
folder_path = "/home/aya/Desktop/GemmaX2-28-9B-v0.1/Augmented Queries Samples"
all_parsed_data = load_all_csv_files(folder_path)

# --- Display summary ---
print("=" * 60)
print("SUMMARY")
print("=" * 60)
for filename, data in all_parsed_data.items():
    print(f"\n{filename}: {len(data)} entries")
    
    # Count entries with target sentences
    with_target = sum(1 for d in data if d.get('target_sentence_english'))
    print(f"  - Entries with target sentence: {with_target}")
    print(f"  - Fields: {set(d.get('field') for d in data)}")

Found 4 CSV files:
  - education_examples.csv
  - technology_examples.csv
  - economic_examples.csv
  - medical_examples.csv

Processing: education_examples.csv
Extracted field: education
Content to parse:
Terms: educational psychology=علم نفس التربية - intrinsic motivation=الدافعية الداخلية - self-efficacy=الكفاءة الذاتية - growth mindset=عقلية النمو - classroom engagement=مشاركة الصف

English: Educational psychology studies how students learn and develop emotionally.
Arabic: يدرس علم نفس التربية كيفية تعلم الطلاب وتطورهم عاطفياً.

English: Teachers often focus on boosting student confidence in learning environments.
Arabic: يركز المعلمون غالباً على تعزيز ثقة الطلاب في بيئات التعلم.

English: Motivation plays a key role in academic achievement.
Arabic: تلعب الدافعية دوراً رئيسياً في الإنجاز الأكاديمي.

English: A positive attitude towards challenges helps learners persist.
Arabic: الموقف الإيجابي تجاه التحديات يساعد المتعلمين على الاستمرار.

English: In educational psychology, intrins

In [5]:
# --- Display detailed example from one file ---
print("=" * 60)
print("DETAILED EXAMPLE (First entry from economic_examples.csv)")
print("=" * 60)

if 'economic_examples.csv' in all_parsed_data:
    example = all_parsed_data['economic_examples.csv'][0]
    
    print(f"\nField: {example.get('field')}")
    print(f"Source File: {example.get('source_file')}")
    print(f"Row Index: {example.get('row_index')}")
    
    print("\n--- Extracted Terms ---")
    for eng, arab in example.get('extracted_terms', []):
        print(f"  {eng} = {arab}")
    
    print("\n--- Few-Shot Examples ---")
    for i, (eng, arab) in enumerate(example.get('few_shot_examples', []), 1):
        print(f"  Example {i}:")
        print(f"    English: {eng}")
        print(f"    Arabic: {arab}")
    
    print("\n--- Target Sentence (to translate) ---")
    print(f"  English: {example.get('target_sentence_english')}")
else:
    print("economic_examples.csv not found in parsed data")

DETAILED EXAMPLE (First entry from economic_examples.csv)

Field: economic
Source File: economic_examples.csv
Row Index: 0

--- Extracted Terms ---
  quantitative easing program = برنامج التسهيل الكمي
  federal funds rate = معدل الأموال الفيدرالية
  inflation targeting framework = إطار استهداف التضخم
  forward guidance communication = تواصل التوجيه التطلعي
  balance sheet normalization = تطبيع الميزانية العمومية

--- Few-Shot Examples ---
  Example 1:
    English: Quantitative easing program increases central-bank asset purchases.
    Arabic: يزيد برنامج التسهيل الكمي من مشتريات البنك المركزي من الأصول.
  Example 2:
    English: Federal funds rate influences short-term borrowing costs.
    Arabic: يؤثر معدل الأموال الفيدرالية على تكاليف الاقتراض قصيرة الأجل.
  Example 3:
    English: Inflation targeting framework anchors inflation expectations.
    Arabic: يرسخ إطار استهداف التضخم توقعات التضخم.
  Example 4:
    English: Balance sheet normalization reduces central-bank holdings gradual

In [6]:
# --- Create a flat list of all entries for easy iteration ---
def get_all_entries_flat(all_parsed_data: Dict[str, List[Dict]]) -> List[Dict]:
    """
    Flatten all parsed data into a single list for easy iteration.
    """
    flat_list = []
    for filename, entries in all_parsed_data.items():
        flat_list.extend(entries)
    return flat_list

# Get flat list
all_entries = get_all_entries_flat(all_parsed_data)
print(f"Total entries across all files: {len(all_entries)}")

# Filter only entries that have a target sentence to translate
entries_to_translate = [e for e in all_entries if e.get('target_sentence_english')]
print(f"Entries with target sentences to translate: {len(entries_to_translate)}")

# --- Show first 5 target sentences ---
print("\n" + "=" * 60)
print("FIRST 5 TARGET SENTENCES TO TRANSLATE")
print("=" * 60)
for i, entry in enumerate(entries_to_translate[:5], 1):
    print(f"\n{i}. [{entry.get('field')}] from {entry.get('source_file')}")
    print(f"   Target: {entry.get('target_sentence_english')[:100]}..."
          if len(entry.get('target_sentence_english', '')) > 100 
          else f"   Target: {entry.get('target_sentence_english')}")

Total entries across all files: 421
Entries with target sentences to translate: 325

FIRST 5 TARGET SENTENCES TO TRANSLATE

1. [education] from education_examples.csv
   Target: In educational psychology, intrinsic motivation enhances self-efficacy and fosters a growth mindset,...

2. [education] from education_examples.csv
   Target: The zone of proximal development requires a scaffolding technique to build cognitive skills through ...

3. [education] from education_examples.csv
   Target: Emotional intelligence in social learning relies on peer interactions to promote empathy development...

4. [education] from education_examples.csv
   Target: Formative assessment employs feedback loops to improve learning outcomes via metacognitive strategie...

5. [education] from education_examples.csv
   Target: Behavior modification uses reinforcement schedules in operant conditioning to apply positive reinfor...


In [7]:
def build_few_shot_prompt(entry: Dict) -> str:
    """
    Build a few-shot translation prompt using the entry's terms and examples.
    """
    prompt_parts = []
    
    # Add instruction header
    prompt_parts.append("Translate the following English sentence to Arabic.")
    prompt_parts.append(f"Domain: {entry.get('field', 'general')}\n")
    
    # Add terminology section if available
    terms = entry.get('extracted_terms', [])
    if terms:
        prompt_parts.append("Use the following terminology:")
        for eng_term, arab_term in terms:
            prompt_parts.append(f"  - {eng_term} = {arab_term}")
        prompt_parts.append("")
    
    # Add few-shot examples
    few_shots = entry.get('few_shot_examples', [])
    if few_shots:
        prompt_parts.append("Examples:")
        for eng, arab in few_shots:
            prompt_parts.append(f"English: {eng}")
            prompt_parts.append(f"Arabic: {arab}")
            prompt_parts.append("")
    
    # Add the target sentence to translate
    target = entry.get('target_sentence_english', '')
    prompt_parts.append("Now translate this sentence:")
    prompt_parts.append(f"English: {target}")
    prompt_parts.append("Arabic:")
    
    return "\n".join(prompt_parts)


def translate_with_few_shot(entry: Dict, model, tokenizer, max_new_tokens: int = 200, num_translations: int = 5) -> List[str]:
    """
    Translate the target sentence using few-shot prompting with terms and examples.
    Returns up to num_translations diverse translations.
    """
    # Build the prompt
    prompt = build_few_shot_prompt(entry)
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate multiple translations using beam search with diverse beam groups
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=num_translations,
        num_return_sequences=num_translations,
        do_sample=False,  # Use beam search for quality
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        early_stopping=True
    )
    
    # Decode all translations
    translations = []
    input_length = inputs['input_ids'].shape[-1]
    
    for i in range(num_translations):
        generated_translation = tokenizer.decode(
            output_ids[i][input_length:],
            skip_special_tokens=True
        )
        # Clean up - take only the first line (the translation)
        translation = generated_translation.strip().split('\n')[0].strip()
        if translation:  # Only add non-empty translations
            translations.append(translation)
    
    return translations


def translate_batch(entries: List[Dict], model, tokenizer, 
                    max_entries: int = None, verbose: bool = True) -> List[Dict]:
    """
    Translate multiple entries and return results with translations.
    """
    results = []
    
    entries_to_process = entries[:max_entries] if max_entries else entries
    total = len(entries_to_process)
    
    for i, entry in enumerate(entries_to_process):
        if verbose:
            print(f"Translating {i+1}/{total}: [{entry.get('field')}] from {entry.get('source_file')}")
        
        # Skip if no target sentence
        if not entry.get('target_sentence_english'):
            if verbose:
                print("  Skipped - no target sentence")
            continue
        
        # Translate - get multiple translations
        translations = translate_with_few_shot(entry, model, tokenizer)
        
        # Store result
        result = {
            'field': entry.get('field'),
            'source_file': entry.get('source_file'),
            'row_index': entry.get('row_index'),
            'english': entry.get('target_sentence_english'),
            'arabic_translations': translations,  # Now a list
            'num_terms': len(entry.get('extracted_terms', [])),
            'num_few_shots': len(entry.get('few_shot_examples', []))
        }
        results.append(result)
        
        if verbose:
            print(f"  English: {result['english'][:80]}...")
            for idx, trans in enumerate(translations, 1):
                trans_preview = trans[:80] + '...' if len(trans) > 80 else trans
                print(f"  Arabic {idx}: {trans_preview}")
            print()
    
    return results


# --- Example: Show what the prompt looks like ---
print("=" * 70)
print("EXAMPLE PROMPT STRUCTURE")
print("=" * 70)
if entries_to_translate:
    example_prompt = build_few_shot_prompt(entries_to_translate[0])
    print(example_prompt)
    print("\n" + "=" * 70)

EXAMPLE PROMPT STRUCTURE
Translate the following English sentence to Arabic.
Domain: education

Use the following terminology:
  - educational psychology = علم نفس التربية
  - intrinsic motivation = الدافعية الداخلية
  - self-efficacy = الكفاءة الذاتية
  - growth mindset = عقلية النمو
  - classroom engagement = مشاركة الصف

Examples:
English: Educational psychology studies how students learn and develop emotionally.
Arabic: يدرس علم نفس التربية كيفية تعلم الطلاب وتطورهم عاطفياً.

English: Teachers often focus on boosting student confidence in learning environments.
Arabic: يركز المعلمون غالباً على تعزيز ثقة الطلاب في بيئات التعلم.

English: Motivation plays a key role in academic achievement.
Arabic: تلعب الدافعية دوراً رئيسياً في الإنجاز الأكاديمي.

English: A positive attitude towards challenges helps learners persist.
Arabic: الموقف الإيجابي تجاه التحديات يساعد المتعلمين على الاستمرار.

English: In educational psychology, intrinsic motivation enhances self-efficacy and fosters a gro

In [8]:
# --- Run translation on a few examples ---
# Start with a small batch to test (adjust max_entries as needed)

print("=" * 70)
print("RUNNING TRANSLATIONS (First 5 entries)")
print("=" * 70)

# Translate first 5 entries as a test
translation_results = translate_batch(
    entries_to_translate, 
    model, 
    tokenizer, 
    max_entries=5, 
    verbose=True
)

print("\n" + "=" * 70)
print("TRANSLATION RESULTS SUMMARY")
print("=" * 70)
for i, result in enumerate(translation_results, 1):
    print(f"\n{i}. [{result['field']}]")
    print(f"   English: {result['english']}")
    for idx, trans in enumerate(result['arabic_translations'], 1):
        print(f"   Arabic {idx}: {trans}")

RUNNING TRANSLATIONS (First 5 entries)
Translating 1/5: [education] from education_examples.csv
  English: In educational psychology, intrinsic motivation enhances self-efficacy and foste...
  Arabic 1: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، مم...
  Arabic 2: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، مم...
  Arabic 3: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، ...
  Arabic 4: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، ...
  Arabic 5: في علم النفس التربوي، تعزز الدافعية الذاتية الكفاءة الذاتية وتعزز عقلية النمو، م...

Translating 2/5: [education] from education_examples.csv
  English: The zone of proximal development requires a scaffolding technique to build cogni...
  Arabic 1: تتطلب منطقة التطور القريب تقنية الإسناد لبناء المهارات المعرفية من خلال إرشاد ال...
  Arabic 2: تتطلب منطقة التطور القريب تقنية الإسناد لبناء المهارات المعرفية من خ

In [9]:
# --- Optional: Translate ALL entries and save to CSV ---
# Uncomment and run this cell to translate all entries

# print("=" * 70)
# print("TRANSLATING ALL ENTRIES (This may take a while...)")
# print("=" * 70)

# all_translation_results = translate_batch(
#     entries_to_translate, 
#     model, 
#     tokenizer, 
#     max_entries=None,  # Process all
#     verbose=True
# )

# # Save to CSV
# results_df = pd.DataFrame(all_translation_results)
# output_path = "/home/aya/Desktop/GemmaX2-28-9B-v0.1/translation_results.csv"
# results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
# print(f"\nResults saved to: {output_path}")
# print(f"Total translations: {len(all_translation_results)}")


# --- Quick function to translate a single entry by index ---
def translate_single(index: int, verbose: bool = True) -> Dict:
    """
    Translate a single entry by its index in entries_to_translate list.
    """
    if index < 0 or index >= len(entries_to_translate):
        print(f"Index {index} out of range. Valid range: 0-{len(entries_to_translate)-1}")
        return None
    
    entry = entries_to_translate[index]
    
    if verbose:
        print(f"Translating entry {index}:")
        print(f"  Field: {entry.get('field')}")
        print(f"  Source: {entry.get('source_file')}")
        print(f"  Terms: {len(entry.get('extracted_terms', []))}")
        print(f"  Few-shots: {len(entry.get('few_shot_examples', []))}")
        print()
    
    translations = translate_with_few_shot(entry, model, tokenizer)
    
    result = {
        'english': entry.get('target_sentence_english'),
        'arabic_translations': translations
    }
    
    if verbose:
        print(f"English: {result['english']}")
        for idx, trans in enumerate(translations, 1):
            print(f"Arabic {idx}: {trans}")
    
    return result


# Example: Translate entry at index 0
print("=" * 70)
print("TRANSLATE SINGLE ENTRY EXAMPLE")
print("=" * 70)
result = translate_single(0)

TRANSLATE SINGLE ENTRY EXAMPLE
Translating entry 0:
  Field: education
  Source: education_examples.csv
  Terms: 5
  Few-shots: 5

English: In educational psychology, intrinsic motivation enhances self-efficacy and fosters a growth mindset, leading to greater classroom engagement.
Arabic 1: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
Arabic 2: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
Arabic 3: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
Arabic 4: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
Arabic 5: في علم النفس التربوي، تعزز الدافعية الذاتية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.


In [10]:
# Load template files with ground truth translations
import pandas as pd

def load_templates(folder_path: str) -> Dict[str, Dict[str, str]]:
    """
    Load all *_translation_template.csv files and create a mapping of
    English sentence -> Arabic ground truth translation for each field.
    """
    templates = {}
    
    # Find all template files
    template_files = glob(os.path.join(folder_path, "*_translation_template.csv"))
    
    print(f"Found {len(template_files)} template files:")
    for f in template_files:
        print(f"  - {os.path.basename(f)}")
    print()
    
    for template_file in template_files:
        filename = os.path.basename(template_file)
        field = filename.replace("_translation_template.csv", "")
        
        print(f"Loading {filename}...")
        df = pd.read_csv(template_file)
        
        # Create mapping: English -> Arabic ground truth
        template_mapping = {}
        for idx, row in df.iterrows():
            english = row.get('English', '').strip()
            arabic = row.get('Arabic', '').strip()
            if english and arabic:
                template_mapping[english] = arabic
        
        templates[field] = template_mapping
        print(f"  Loaded {len(template_mapping)} translation pairs\n")
    
    return templates

# Load all templates
template_path = "/home/aya/Desktop/GemmaX2-28-9B-v0.1/Augmented Queries Samples"
ground_truth_templates = load_templates(template_path)

print("=" * 70)
print("GROUND TRUTH TEMPLATES LOADED")
print("=" * 70)
for field, mapping in ground_truth_templates.items():
    print(f"{field}: {len(mapping)} translation pairs")

Found 4 template files:
  - education_translation_template.csv
  - medical_translation_template.csv
  - economic_translation_template.csv
  - technology_translation_template.csv

Loading education_translation_template.csv...
  Loaded 90 translation pairs

Loading medical_translation_template.csv...
  Loaded 55 translation pairs

Loading economic_translation_template.csv...
  Loaded 90 translation pairs

Loading technology_translation_template.csv...
  Loaded 90 translation pairs

GROUND TRUTH TEMPLATES LOADED
education: 90 translation pairs
medical: 55 translation pairs
economic: 90 translation pairs
technology: 90 translation pairs


In [14]:
# Note: COMET requires a pre-trained model which will be downloaded on first use
try:
    from comet import download_model, load_from_checkpoint
    comet_available = True
    comet_model = None  # Will be loaded on first use
except:
    comet_available = False
    comet_model = None
    print("Warning: COMET library not available")

In [ ]:
# Import evaluation libraries and implement evaluation functions
from bert_score import score as bert_score
from sacrebleu.metrics import CHRF
import numpy as np



def get_comet_model():
    """Load COMET model on first use (lazy loading)."""
    global comet_model
    if comet_model is None and comet_available:
        try:
            model_path = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(model_path)
            print("✓ COMET model loaded successfully")
        except Exception as e:
            print(f"Warning: Could not load COMET model: {e}")
    return comet_model

def evaluate_translation(reference: str, hypothesis: str, source: str = "", src_lang: str = "en", tgt_lang: str = "ar") -> Dict[str, float]:
    """
    Evaluate a single translation using BERTScore, CHRF++, and COMET.
    """
    scores = {}
    
    # BERTScore
    try:
        P, R, F1 = bert_score([hypothesis], [reference], lang=tgt_lang, verbose=False)
        scores['bert_score_f1'] = F1.item()
    except Exception as e:
        scores['bert_score_f1'] = None
        print(f"  BERTScore error: {e}")
    
    # CHRF++
    try:
        chrf_metric = CHRF(word_order=2)  # CHRF++ uses word order=2
        chrf_score = chrf_metric.corpus_score([hypothesis], [[reference]])
        scores['chrf_plus_plus'] = chrf_score.score  # Already normalized to 0-1 by sacrebleu
    except Exception as e:
        scores['chrf_plus_plus'] = None
        print(f"  CHRF++ error: {e}")
    
    # COMET
    try:
        comet = get_comet_model()
        if comet is not None:
            # COMET expects [{'src': source, 'mt': hypothesis, 'ref': reference}]
            data = [{'src': source, 'mt': hypothesis, 'ref': reference}]
            comet_scores = comet.predict(data, batch_size=1, gpus=1)
            scores['comet'] = comet_scores['scores'][0]
        else:
            scores['comet'] = None
    except Exception as e:
        scores['comet'] = None
        print(f"  COMET error: {e}")
    
    return scores

def evaluate_batch_translations(entries: List[Dict], translations_results: List[Dict], 
                                ground_truth_templates: Dict[str, Dict[str, str]],
                                verbose: bool = True) -> List[Dict]:
    """
    Evaluate all generated translations against ground truth templates.
            scores = evaluate_translation(ground_truth, translation, source=english_sentence)
    """
    evaluation_results = []
    
    for result in translations_results:
        field = result.get('field')
        print(f"\nEvaluating translations for field: {field}")
        english_sentence = result.get('english')
        generated_translations = result.get('arabic_translations', [])
                'chrf_plus_plus': scores.get('chrf_plus_plus'),
                'comet': scores.get('comet')
            }
            evaluation_results.append(eval_entry)
            
            if verbose:
                print(f"\nTranslation {idx}: {translation}")
                if scores.get('bert_score_f1') is not None:
                    print(f"  BERTScore F1: {scores['bert_score_f1']:.4f}")
                if scores.get('chrf_plus_plus') is not None:
                    print(f"  CHRF++: {scores['chrf_plus_plus']:.4f}")
                if scores.get('comet') is not None:
                    print(f"  COMET: {scores['comet']:.4f}")
    

    return evaluation_results
            print(f"Ground Truth (Arabic): {ground_truth}")print("Evaluation functions ready!")


            print(f"{'='*70}")

print("Evaluation functions ready!")
            return evaluation_results

        # Evaluate each generated translation    

        for idx, translation in enumerate(generated_translations, 1):                    print(f"  CHRF++: {scores['chrf_plus_plus']:.4f}")

            scores = evaluate_translation(ground_truth, translation)                if scores.get('chrf_plus_plus') is not None:

                                print(f"  BERTScore F1: {scores['bert_score_f1']:.4f}")

            eval_entry = {                if scores.get('bert_score_f1') is not None:

                'field': field,                print(f"\nTranslation {idx}: {translation}")

                'english': english_sentence,            if verbose:

                'ground_truth_arabic': ground_truth,            

                'generated_translation': translation,            evaluation_results.append(eval_entry)

                'translation_rank': idx,            }

                'bert_score_f1': scores.get('bert_score_f1'),                'chrf_plus_plus': scores.get('chrf_plus_plus')

Evaluation functions ready!


In [ ]:
# Evaluate the translation results from earlier
print("=" * 70)
print("EVALUATING TRANSLATIONS")
print("=" * 70)

# Use the translation_results from the earlier cell
evaluation_results = evaluate_batch_translations(
    entries_to_translate,
    translation_results,
    ground_truth_templates,
    verbose=True
)

print("\n" + "=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)

if evaluation_results:
    # Convert to DataFrame for analysis
    eval_df = pd.DataFrame(evaluation_results)
    
    print(f"\nTotal evaluations: {len(eval_df)}")
    print(f"Fields covered: {eval_df['field'].unique().tolist()}")
    
    # Summary statistics
    print("\n" + "=" * 70)
    print("METRIC STATISTICS")
    print("=" * 70)
    
    if 'bert_score_f1' in eval_df.columns:
        valid_bert = eval_df['bert_score_f1'].dropna()
        if len(valid_bert) > 0:
            print(f"\nBERTScore F1:")
            print(f"  Mean: {valid_bert.mean():.4f}")
            print(f"  Median: {valid_bert.median():.4f}")
            print(f"  Std Dev: {valid_bert.std():.4f}")
            print(f"  Min: {valid_bert.min():.4f}")
            print(f"  Max: {valid_bert.max():.4f}")
    
    if 'chrf_plus_plus' in eval_df.columns:
        valid_chrf = eval_df['chrf_plus_plus'].dropna()
        if len(valid_chrf) > 0:
            print(f"\nCHRF++:")
            print(f"  Mean: {valid_chrf.mean():.4f}")
            print(f"  Median: {valid_chrf.median():.4f}")
            print(f"  Std Dev: {valid_chrf.std():.4f}")
            print(f"  Min: {valid_chrf.min():.4f}")
            print(f"  Max: {valid_chrf.max():.4f}")
    
    if 'comet' in eval_df.columns:
        valid_comet = eval_df['comet'].dropna()
        if len(valid_comet) > 0:
            print(f"\nCOMET:")
            print(f"  Mean: {valid_comet.mean():.4f}")
            print(f"  Median: {valid_comet.median():.4f}")
            print(f"  Std Dev: {valid_comet.std():.4f}")
            print(f"  Min: {valid_comet.min():.4f}")
            print(f"  Max: {valid_comet.max():.4f}")
    
    # Per-field analysis
    print("\n" + "=" * 70)
    print("PER-FIELD ANALYSIS")
    print("=" * 70)
    
    for field in eval_df['field'].unique():
        field_data = eval_df[eval_df['field'] == field]
        print(f"\n{field.upper()}:")
        print(f"  Entries: {len(field_data)}")
        
        bert_scores = field_data['bert_score_f1'].dropna()
        if len(bert_scores) > 0:
            print(f"  BERTScore F1 Mean: {bert_scores.mean():.4f}")
        
        chrf_scores = field_data['chrf_plus_plus'].dropna()
        if len(chrf_scores) > 0:
            print(f"  CHRF++ Mean: {chrf_scores.mean():.4f}")
    
    # Best translations by metric
    print("\n" + "=" * 70)
    print("TOP 5 TRANSLATIONS BY BERTSCORE F1")
    print("=" * 70)
    
    if 'bert_score_f1' in eval_df.columns:
        top_bert = eval_df.nlargest(5, 'bert_score_f1')[['field', 'english', 'generated_translation', 'bert_score_f1']]
        for idx, row in top_bert.iterrows():
            print(f"\n[{row['field']}] Score: {row['bert_score_f1']:.4f}")
            print(f"  English: {row['english'][:70]}")
            print(f"  Arabic: {row['generated_translation'][:70]}")
    
    print("\n" + "=" * 70)
    print("TOP 5 TRANSLATIONS BY CHRF++")
    print("=" * 70)
    
    if 'chrf_plus_plus' in eval_df.columns:
        top_chrf = eval_df.nlargest(5, 'chrf_plus_plus')[['field', 'english', 'generated_translation', 'chrf_plus_plus']]
        for idx, row in top_chrf.iterrows():
            print(f"\n[{row['field']}] Score: {row['chrf_plus_plus']:.4f}")
            print(f"  English: {row['english'][:70]}")
            print(f"  Arabic: {row['generated_translation'][:70]}")
else:
    print("No evaluation results generated. Check if ground truth templates match the generated sentences.")

EVALUATING TRANSLATIONS

Evaluating translations for field: education

Evaluating: [education]
English: In educational psychology, intrinsic motivation enhances self-efficacy and fosters a growth mindset, leading to greater classroom engagement.
Ground Truth (Arabic): في علم النفس التربوي، يعزز الدافع الداخلي الكفاءة الذاتية ويعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الفصل الدراسي.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


Translation 1: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
  BERTScore F1: 0.9179
  CHRF++: 66.4196

Translation 2: في علم نفس التربية، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
  BERTScore F1: 0.9072
  CHRF++: 64.3324

Translation 3: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
  BERTScore F1: 0.9412
  CHRF++: 74.2661

Translation 4: في علم النفس التربوي، تعزز الدافعية الداخلية الكفاءة الذاتية وتنمي عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
  BERTScore F1: 0.9310
  CHRF++: 72.1876

Translation 5: في علم النفس التربوي، تعزز الدافعية الذاتية الكفاءة الذاتية وتعزز عقلية النمو، مما يؤدي إلى مشاركة أكبر في الصف.
  BERTScore F1: 0.9387
  CHRF++: 71.5391

Evaluating translations for field: education

Evaluating: [education]
English: The zone of proximal development requires a scaffolding technique to b

In [13]:
# Save evaluation results to CSV
if evaluation_results:
    results_df = pd.DataFrame(evaluation_results)
    
    output_path = "/home/aya/Desktop/GemmaX2-28-9B-v0.1/evaluation_results.csv"
    results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print("=" * 70)
    print("EVALUATION RESULTS SAVED")
    print("=" * 70)
    print(f"Saved to: {output_path}")
    print(f"Total rows: {len(results_df)}")
    print(f"\nColumns: {', '.join(results_df.columns.tolist())}")
else:
    print("No results to save.")

EVALUATION RESULTS SAVED
Saved to: /home/aya/Desktop/GemmaX2-28-9B-v0.1/evaluation_results.csv
Total rows: 25

Columns: field, english, ground_truth_arabic, generated_translation, translation_rank, bert_score_f1, chrf_plus_plus
